In [ ]:
"""
ICA (Independent Component Analysis)
- Separates mixed EEG into independent sources
- Artifacts often become isolated components
- Components must be manually inspected
"""

print("Running ICA (this may take 1-2 minutes)...")

ica = mne.preprocessing.ICA(
    n_components=0.99,      # retain 99% PCA variance
    method='fastica',       # stable convergence on this dataset
    random_state=42,
    max_iter=2000
)

ica.fit(raw_clean)

print("Converged?", ica.n_iter_ < 2000)
print("Iterations:", ica.n_iter_)

print(f"ICA complete")
print(f"ICA components retained: {ica.n_components_}")
print(f"ICA iterations: {ica.n_iter_}")

# Inspect candidate artifact components

ica.plot_properties(
    raw_clean,
    picks=[0, 1, 3, 5, 6, 8, 12, 13, 15, 16]
)
plt.show()

# Optional overview plots

ica.plot_components(show=False)
plt.show()

ica.plot_sources(
    raw_clean,
    start=60,
    stop=120,
    title='ICA Sources (t=60-120s)',
    show=False
)
plt.show()

"""
Artifact Guide
--------------
Blink -> frontal topography + strong low frequencies
Muscle -> peripheral topography + high frequencies
Eye movement-> left/right frontal pattern
Neural -> smooth spatial pattern + physiologic spectrum
"""

# Components chosen after inspection

ica.exclude = [3, 5, 6, 8, 12, 13, 15, 16]

print(f"Components marked for removal: {ica.exclude}")

# APPLY ICA

raw_ica = raw_clean.copy()

if len(ica.exclude) > 0:

    ica.apply(raw_ica)

    print(f"Removed {len(ica.exclude)} components")

    print("\nShowing EEG before ICA...")

    raw_clean.plot(
        n_channels=20,
        duration=20,
        start=60,
        scalings=dict(eeg=20e-6),
        title='Before ICA'
    )

    print("\nShowing EEG after ICA...")

    raw_ica.plot(
        n_channels=20,
        duration=20,
        start=60,
        scalings=dict(eeg=20e-6),
        title='After ICA'
    )

    # SIGNAL REMOVED BY ICA


    print("\nShowing signal removed by ICA...")

    raw_diff = raw_clean.copy()

    raw_diff._data = (
        raw_clean.get_data()
        - raw_ica.get_data()
    )

    raw_diff.plot(
        n_channels=20,
        duration=20,
        start=60,
        scalings=dict(eeg=20e-6),
        title='Signal Removed By ICA'
    )


    # RECONSTRUCTION CHECK
    removed_rms = np.sqrt(
        np.mean(raw_diff.get_data() ** 2)
    )

    original_rms = np.sqrt(
        np.mean(raw_clean.get_data() ** 2)
    )

    print(f"Removed signal RMS : {removed_rms:.3e}")
    print(f"Original signal RMS: {original_rms:.3e}")
    print(
        f"Removed fraction   : "
        f"{100 * removed_rms / original_rms:.2f}%"
    )

else:
    print("No components removed")

# PSD COMPARISON

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw_clean.compute_psd(
    fmax=40
).plot(
    picks='eeg',
    axes=axes[0],
    show=False
)

axes[0].set_title("Before ICA")

raw_ica.compute_psd(
    fmax=40
).plot(
    picks='eeg',
    axes=axes[1],
    show=False
)

axes[1].set_title(
    f"After ICA ({len(ica.exclude)} removed)"
)

plt.suptitle("ICA Effect on PSD")
plt.tight_layout()
plt.show()

# ALPHA PRESERVATION CHECK
# HydroCel-129 posterior electrodes

alpha_channels = posterior

psd_before = raw_clean.compute_psd(
    fmin=1,
    fmax=40,
    picks=alpha_channels
)

psd_after = raw_ica.compute_psd(
    fmin=1,
    fmax=40,
    picks=alpha_channels
)

freqs = psd_before.freqs

before_data = psd_before.get_data().mean(axis=0)
after_data = psd_after.get_data().mean(axis=0)

alpha_idx = np.where(
    (freqs >= 8) & (freqs <= 13)
)[0]

alpha_before = before_data[alpha_idx]
alpha_after = after_data[alpha_idx]

peak_before = np.max(alpha_before)
peak_after = np.max(alpha_after)

peak_freq_before = freqs[
    alpha_idx[np.argmax(alpha_before)]
]

peak_freq_after = freqs[
    alpha_idx[np.argmax(alpha_after)]
]

print("POSTERIOR ALPHA PRESERVATION")
print(f"Posterior channels used: {len(alpha_channels)}")

print(
    f"Alpha peak before ICA : "
    f"{peak_freq_before:.2f} Hz"
)

print(
    f"Alpha peak after ICA  : "
    f"{peak_freq_after:.2f} Hz"
)

print(
    f"Peak power before ICA : "
    f"{peak_before:.3e}"
)

print(
    f"Peak power after ICA  : "
    f"{peak_after:.3e}"
)

percent_change = (
    100 * (peak_after - peak_before)
    / peak_before
)

alpha_change = percent_change

print(f"Percent change : {percent_change:.2f}%")



from scipy.integrate import simpson

alpha_power_before = simpson(
    alpha_before,
    freqs[alpha_idx]
)

alpha_power_after = simpson(
    alpha_after,
    freqs[alpha_idx]
)

alpha_power_change = (
    100 * (alpha_power_after - alpha_power_before)
    / alpha_power_before
)

print("\nALPHA BAND POWER")
print("---------------------")
print(f"Before ICA : {alpha_power_before:.3e}")
print(f"After ICA  : {alpha_power_after:.3e}")
print(f"Change     : {alpha_power_change:.2f}%")

if alpha_change > -1:
    print("✅ Alpha largely preserved")

elif alpha_change > -3:
    print("⚠️ Mild alpha reduction")

else:
    print("❌ Significant alpha reduction")

print("=" * 60)


plt.figure(figsize=(8, 5))

plt.plot(
    freqs,
    before_data,
    label="Before ICA"
)

plt.plot(
    freqs,
    after_data,
    label="After ICA"
)

plt.axvspan(
    8,
    13,
    alpha=0.2,
    label="Alpha Band"
)

plt.xlim(1, 40)

plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD")

plt.title("Posterior Alpha Preservation")

plt.legend()
plt.tight_layout()
plt.show()



print("\nFINAL ICA SUMMARY")

print(f"ICA method           : {ica.method}")
print(f"Components kept      : {ica.n_components_}")
print(f"Components removed   : {len(ica.exclude)}")
print(f"Iterations           : {ica.n_iter_}")

print(
    f"Alpha peak change (%) : "
    f"{alpha_change:.2f}"
)

print(
    f"Alpha band power (%)  : "
    f"{alpha_power_change:.2f}"
)

print("ICA components:", ica.n_components_)
print("PCA components:", ica.pca_components_.shape[0])

if hasattr(ica, "pca_explained_variance_") and \
   ica.pca_explained_variance_ is not None:

    evr = ica.pca_explained_variance_

    retained = (
        np.sum(evr[:ica.n_components_])
        / np.sum(evr)
    )

    print(
        f"PCA variance retained: "
        f"{retained:.2%}"
    )

In [ ]:
"""
MONTAGE VERIFICATION
Confirm that HydroCel-129 electrode coordinates are available.
"""

print("\n" + "="*60)
print("MONTAGE VERIFICATION")
print("="*60)

montage = raw.get_montage()

if montage is None:
    raise RuntimeError(
        "No montage found. Electrode coordinates are required for spatial analysis."
    )

print(f"Montage loaded: {montage}")
print(f"Total EEG channels: {len(raw.ch_names)}")

print("\nHydroCel-129 montage successfully attached.")
print("Electrode coordinate information is available.")
print("="*60)

In [ ]:
"""
SENSOR LAYOUT VISUALIZATION
Visual verification of HydroCel-129 electrode locations.
Nose is at the top, posterior electrodes are toward the bottom.
"""

raw.plot_sensors(
    kind="topomap",
    show_names=False
)

plt.show()

In [ ]:
"""
POSTERIOR CHANNEL IDENTIFICATION

Posterior electrodes are identified using the MNE head coordinate system:
    y > 0  -> anterior (front)
    y < 0  -> posterior (back)

These channels will be used for alpha-band validation after ICA.
"""

print("\n" + "="*60)
print("POSTERIOR CHANNEL IDENTIFICATION")
print("="*60)

montage = raw.get_montage()

if montage is None:
    raise RuntimeError(
        "No montage found. Cannot identify posterior channels."
    )

positions = montage.get_positions()["ch_pos"]

posterior = []

for ch in raw.ch_names:

    if ch not in positions:
        continue

    x, y, z = positions[ch]

    if y < -0.04:
        posterior.append(ch)

print(f"Posterior channels identified: {len(posterior)}")

print("\nPosterior electrodes:")
print(posterior)

print("="*60)

In [ ]:
"""
Inspect BIDS events file.
RestingState recordings may contain few or no task events.
"""

events_tsv = DATA_ROOT / SUBJECT / "eeg" / f"{SUBJECT}_task-{TASK}_events.tsv"

print("\n" + "="*70)
print("EVENTS FILE INSPECTION")
print("="*70)

if events_tsv.exists():

    events_df = pd.read_csv(events_tsv, sep="\t")

    print(f"Events file found: {events_tsv.name}")
    print(f"Number of events: {len(events_df)}")
    print(f"Columns: {list(events_df.columns)}")

    print("\nFirst 5 rows:")
    print(events_df.head().to_string(index=False))

else:
    print("No events.tsv file found (normal for some RestingState recordings)")

In [ ]:
"""
Inspect HBN quality-control table.
Used to verify whether the selected subject contains usable EEG data.
"""

quality_tsv = DATA_ROOT / "code" / "RestingState_quality_table.tsv"

print("\n" + "="*70)
print("QUALITY TABLE INSPECTION")
print("="*70)

if quality_tsv.exists():

    qdf = pd.read_csv(quality_tsv, sep=None, engine="python")

    print(f"Quality table found: {quality_tsv.name}")
    print(f"Subjects listed: {len(qdf)}")

    subj_id = SUBJECT.replace("sub-", "")

    subject_col = qdf.columns[0]

    row = qdf[
        qdf[subject_col].astype(str).str.strip() == subj_id
    ]

    if not row.empty:

        print(f"\n{SUBJECT} quality entry:")
        print(row.to_string(index=False))

    else:
        print(f"\n{SUBJECT} not found in quality table")

else:
    print("Quality table not found (expected in some mini releases)")

In [ ]:
"""
Run this ONLY after validating the pipeline on one subject.

Batch preprocessing pipeline:
- HydroCel montage
- 1–40 Hz filtering
- Average reference
- Bad channel interpolation
- Fixed-length epoching

NOTE:
ICA is intentionally NOT included here because
artifact components must be manually inspected
for each subject before removal.
"""

TARGET_TASK = "RestingState"

output_dir = DATA_ROOT / "preprocessed_epochs"
output_dir.mkdir(exist_ok=True)

# ============================================================
# Find all subjects
# ============================================================

subject_dirs = sorted(
    [
        p for p in DATA_ROOT.iterdir()
        if p.is_dir() and p.name.startswith("sub-")
    ]
)

results = []

print(f"Processing {len(subject_dirs)} subjects...\n")

# ============================================================
# Process each subject
# ============================================================

for subj_dir in subject_dirs:

    subj_id = subj_dir.name

    bdf_path = (
        subj_dir
        / "eeg"
        / f"{subj_id}_task-{TARGET_TASK}_eeg.bdf"
    )

    if not bdf_path.exists():
        results.append({
            "subject": subj_id,
            "status": "NO_FILE"
        })
        continue

    try:

        # ----------------------------------------------------
        # Load EEG
        # ----------------------------------------------------
        r = mne.io.read_raw_bdf(
            bdf_path,
            preload=True,
            verbose=False
        )

        # ----------------------------------------------------
        # HydroCel montage
        # ----------------------------------------------------
        r.set_montage(
            mne.channels.make_standard_montage(
                "GSN-HydroCel-129"
            ),
            on_missing="warn",
            verbose=False
        )

        # ----------------------------------------------------
        # Bandpass filter
        # ----------------------------------------------------
        r.filter(
            l_freq=1.0,
            h_freq=40.0,
            method="iir",
            iir_params={
                "order": 4,
                "ftype": "butter"
            },
            verbose=False
        )

        # ----------------------------------------------------
        # Average reference
        # ----------------------------------------------------
        r.set_eeg_reference(
            "average",
            projection=False,
            verbose=False
        )

        # ----------------------------------------------------
        # Bad channel detection
        # ----------------------------------------------------
        data = r.get_data()

        variances = np.var(data, axis=1)

        mean_v = np.mean(variances)
        std_v = np.std(variances)

        bads = [
            r.ch_names[i]
            for i in range(len(variances))
            if abs(variances[i] - mean_v) > 3 * std_v
        ]

        if bads:
            r.info["bads"] = bads

            r.interpolate_bads(
                reset_bads=True,
                verbose=False
            )

        # ----------------------------------------------------
        # Epoching
        # ----------------------------------------------------
        ep = mne.make_fixed_length_epochs(
            r,
            duration=4.0,
            overlap=0.0,
            preload=True,
            verbose=False
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------
        out_file = (
            output_dir
            / f"{subj_id}_task-{TARGET_TASK}_clean_epo.fif"
        )

        ep.save(
            out_file,
            overwrite=True,
            verbose=False
        )

        results.append({
            "subject": subj_id,
            "n_epochs": len(ep),
            "bad_channels": len(bads),
            "status": "OK"
        })

        print(
            f"{subj_id}: "
            f"{len(ep)} epochs | "
            f"{len(bads)} bad channels"
        )

    except Exception as e:

        results.append({
            "subject": subj_id,
            "status": f"ERROR: {str(e)[:60]}"
        })

        print(f"{subj_id}: ERROR")

# ============================================================
# Save summary
# ============================================================

results_df = pd.DataFrame(results)

summary_file = output_dir / "preprocessing_summary.csv"

results_df.to_csv(
    summary_file,
    index=False
)

print("\n" + "=" * 60)
print("BATCH PREPROCESSING COMPLETE")
print("=" * 60)

print(f"Output directory: {output_dir}")
print(f"Summary file: {summary_file}")

print("\nProcessing summary:")
print(results_df.to_string(index=False))